<a href="https://colab.research.google.com/github/HridhayP/HridhayP.github.io/blob/main/python_notebooks/hw5_bfro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW5: Bigfoot Sightings (BFRO) with Altair
IS 445 — Hridhay Prabahar

Data: [BFRO reports (fall 2022)](https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv)

In [1]:
import pandas as pd
import altair as alt
from vega_datasets import data

## Read the data from URL

In [2]:
url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv'
bfro = pd.read_csv(url)
print(bfro.shape)
bfro.head()

(4747, 29)


,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


## Data transformations
- Keep only the columns the plots use (drops the long free-text report columns so the exported JSON stays small).
- Parse `date` into a datetime and pull out `year`.
- **Map data:** drop reports with no coordinates/date, keep 1950 onward (a handful of 1800s reports would squash the timeline), and drop one mis-entered Washington point with a Pacific Ocean longitude.
- **State data:** use all reports, drop `Unknown` season, count reports per state × season, keep the top 15 states.

In [3]:
cols = ['state', 'county', 'season', 'classification', 'latitude', 'longitude', 'date']
df = bfro[cols].copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year

# map + timeline data
geo = df.dropna(subset=['latitude', 'longitude', 'year']).copy()
geo = geo[geo['year'] >= 1950]
# one Washington report has longitude -167 (in the Pacific), a data-entry error
bad = (~geo['state'].isin(['Alaska', 'Hawaii'])) & (geo['longitude'] < -130)
geo = geo[~bad]
geo['year'] = geo['year'].astype(int)
geo['date'] = geo['date'].dt.strftime('%Y-%m-%d')
print(geo.shape)

# state x season counts (all reports, known season only)
known = df[df['season'] != 'Unknown']
top_states = known['state'].value_counts().head(15).index
state_season = (known[known['state'].isin(top_states)]
                .groupby(['state', 'season']).size()
                .reset_index(name='reports'))
state_season.head()

(3780, 8)


,state,season,reports
0,California,Fall,99
1,California,Spring,66
2,California,Summer,191
3,California,Winter,41
4,Colorado,Fall,44


## Plot 1 (interactive): Where and when sightings happen
Drag across the timeline to filter the map to those years. Click a class in the legend to isolate it (shift-click for multiple).

In [4]:
brush = alt.selection_interval(encodings=['x'])
cls = alt.selection_point(fields=['classification'], bind='legend')

class_colors = alt.Scale(domain=['Class A', 'Class B', 'Class C'],
                         range=['#d95f02', '#1b9e77', '#7570b3'])

states = alt.topo_feature(data.us_10m.url, 'states')
background = alt.Chart(states).mark_geoshape(
    fill='#eeeeee', stroke='white'
).properties(width=700, height=420).project('albersUsa')

points = alt.Chart(geo).mark_circle(size=14).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    color=alt.Color('classification:N', scale=class_colors, title='Report class'),
    opacity=alt.condition(cls, alt.value(0.7), alt.value(0.03)),
    tooltip=['state:N', 'county:N', 'date:N', 'season:N', 'classification:N']
).add_params(cls).transform_filter(brush)

timeline = alt.Chart(geo).mark_bar(color='#555555').encode(
    x=alt.X('year:Q', bin=alt.Bin(step=1), title='Year of sighting', axis=alt.Axis(format='d')),
    y=alt.Y('count():Q', title='Reports')
).add_params(brush).transform_filter(cls).properties(width=700, height=120)

chart1 = alt.vconcat(
    background + points,
    timeline
).properties(title='BFRO Bigfoot sightings, 1950–2018 (drag the timeline to filter the map)')
chart1

alt.VConcatChart(...)

## Plot 2: Which states report the most, and in what season

In [5]:
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
season_colors = alt.Scale(domain=season_order,
                          range=['#4e79a7', '#59a14f', '#f28e2b', '#9c755f'])

chart2 = alt.Chart(state_season).mark_bar().encode(
    y=alt.Y('state:N', sort='-x', title=None),
    x=alt.X('sum(reports):Q', title='Number of reports'),
    color=alt.Color('season:N', scale=season_colors, sort=season_order, title='Season'),
    order=alt.Order('season_rank:Q'),
    tooltip=['state:N', 'season:N', 'reports:Q']
).transform_calculate(
    season_rank="indexof(['Winter','Spring','Summer','Fall'], datum.season)"
).properties(width=600, height=400,
             title='Top 15 states by BFRO reports, split by season')
chart2

alt.Chart(...)

## Export for the Jekyll page
Upload these two files to `assets/json/` in your github.io repo.

In [6]:
chart1.save('bfro_map.json')
chart2.save('bfro_states.json')

# download from Colab
try:
    from google.colab import files
    files.download('bfro_map.json')
    files.download('bfro_states.json')
except ImportError:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>